# Apache Airflow 面试速成：从 0 到 Senior

> 📅 Updated: 2026-03 | Covers Airflow 2.x & 3.0
> 🎯 Target: Data Engineer NG → Senior 面试

---

## 1. 核心概念 (Core Concepts)

### 1.1 基本术语

| 术语 | 定义 | 面试要点 |
|------|------|----------|
| **DAG** (Directed Acyclic Graph) | 有向无环图，定义 task 之间的依赖和执行顺序 | DAG 是逻辑编排，不是数据处理本身 |
| **Task** | DAG 中的一个执行单元 | Task 是 Operator 的实例化 |
| **Operator** | 定义 task 做什么的模板类 | 分 Action / Transfer / Sensor 三类 |
| **Task Instance** | 某个 task 在某个 `logical_date` 的一次具体运行 | 有自己的状态、日志、重试记录 |
| **DAG Run** | 某个 DAG 在某个 `logical_date` 的一次完整执行 | 可以手动 trigger 或 schedule trigger |
| **XCom** | Task 之间传递小数据的机制 (Cross-Communication) | 默认存 metadata DB，不适合传大数据（<48KB推荐） |
| **Connection** | 存储外部系统连接信息（host, port, credentials） | 通过 UI / env var / Secrets Backend 管理 |
| **Variable** | 全局键值对配置 | 不要在 DAG 文件顶层读 Variable（会拖慢 scheduler parsing） |
| **Pool** | 限制并发 task 数量的资源槽 | 用于保护外部系统不被打爆 |
| **Hook** | 与外部系统交互的接口抽象 | Operator 内部调用 Hook；可单独用于自定义逻辑 |

### 1.2 DAG 生命周期

```
DAG 文件 (.py)
    │
    ▼
[Scheduler] ── parse DAG files (每 min_file_process_interval 扫一次)
    │
    ▼
[DagBag] ── 注册 DAG 到 metadata DB
    │
    ▼
[Scheduler] ── 根据 schedule / timetable / asset 创建 DAG Run
    │
    ▼
[Scheduler] ── 解析依赖 → 把 ready tasks 推入 Executor
    │
    ▼
[Executor] ── 分发到 Worker 执行
    │
    ▼
[Worker] ── 执行 task → 更新状态到 metadata DB
    │
    ▼
[Scheduler] ── 检查下游依赖 → 循环直到 DAG Run 完成
```

### 1.3 Scheduling 关键概念

| 概念 | Airflow 2.x | Airflow 3.0 |
|------|-------------|-------------|
| 调度间隔 | `schedule_interval` (cron / timedelta / preset) | `schedule` 参数，支持 cron, timedelta, **timetable**, **Asset** |
| 逻辑日期 | `execution_date`（实际是区间起点，名字误导） | 改名为 `logical_date`，语义更清晰 |
| 数据区间 | `data_interval_start` / `data_interval_end` | 同上，推荐用这对参数 |
| Catchup | `catchup=True` 时自动回填历史 DAG Run | 同上 |
| Backfill | CLI: `airflow dags backfill -s ... -e ...` | 同上，但 3.0 提供更好的 backfill API |

> ⚠️ **经典面试题**：`execution_date` 为什么不是"执行时间"？
> 答：它是 data interval 的起点。比如 daily DAG 的 `execution_date=2026-03-30` 实际在 `2026-03-31` 才运行，处理的是 3/30 的数据。

---

## 2. Operator 类型 (Operator Types)

### 2.1 常用 Operator 速查

| Operator | 用途 | 示例场景 |
|----------|------|----------|
| `BashOperator` | 执行 shell 命令 | `dbt run`, `spark-submit`, 脚本调用 |
| `PythonOperator` | 执行 Python callable | ETL 逻辑、API 调用、数据校验 |
| `@task` (TaskFlow) | PythonOperator 的装饰器语法 | 推荐写法，自动处理 XCom |
| `EmptyOperator` | 占位 / 分支汇聚节点 | DAG 中的 join 点（原 DummyOperator） |
| `BranchPythonOperator` | 动态选择下游分支 | 根据条件执行不同路径 |
| `ShortCircuitOperator` | 条件为 False 时跳过下游所有 task | 数据检查不通过就终止 |
| `TriggerDagRunOperator` | 触发另一个 DAG | 跨 DAG 编排 |
| `ExternalTaskSensor` | 等待另一个 DAG 的 task 完成 | 跨 DAG 依赖 |
| `S3KeySensor` | 等待 S3 文件出现 | 数据到达触发 |
| `SqlSensor` | 等待 SQL 查询返回 True | 上游表数据 ready |
| `BigQueryInsertJobOperator` | 提交 BigQuery job | GCP 数据仓库 ETL |
| `SparkSubmitOperator` | 提交 Spark job | 大数据处理 |
| `KubernetesPodOperator` | 在 K8s Pod 中运行任意容器 | 隔离环境、不同依赖 |

### 2.2 Sensor 深入

```python
# Sensor 的核心参数
sensor_task = S3KeySensor(
    task_id="wait_for_file",
    bucket_key="s3://bucket/path/file.parquet",
    mode="poke",          # poke | reschedule
    poke_interval=300,    # 每 5 min 检查一次
    timeout=3600,         # 1 小时超时
    soft_fail=True,       # 超时标记 skipped 而非 failed
)
```

| Mode | 行为 | 适用场景 |
|------|------|----------|
| `poke` | 占着 worker slot 持续检查 | 预期很快到达（<5 min） |
| `reschedule` | 检查后释放 slot，下次再调度 | 长时间等待（>5 min），节省资源 |

### 2.3 TaskFlow API (Airflow 2.0+)

```python
from airflow.decorators import dag, task
from datetime import datetime

@dag(schedule="@daily", start_date=datetime(2026, 1, 1), catchup=False)
def my_etl():

    @task
    def extract() -> dict:
        return {"data": [1, 2, 3]}

    @task
    def transform(raw: dict) -> dict:
        return {"data": [x * 2 for x in raw["data"]]}

    @task
    def load(transformed: dict):
        print(f"Loading {transformed}")

    raw = extract()
    transformed = transform(raw)
    load(transformed)

my_etl()  # 实例化 DAG
```

> **面试考点**：TaskFlow 的 `@task` 自动序列化返回值到 XCom，下游 task 自动 pull。比传统 `ti.xcom_push/pull` 简洁得多。

---

## 3. 架构与 Executor (Architecture & Executors)

### 3.1 核心组件

```
┌────────────────────────────────────────────────────┐
│                   Metadata DB                       │
│              (PostgreSQL / MySQL)                    │
│   存储: DAG 定义、DAG Run、Task Instance、XCom、    │
│         Connection、Variable、日志引用              │
└──────────┬──────────────┬──────────────┬───────────┘
           │              │              │
    ┌──────▼──────┐ ┌─────▼─────┐ ┌─────▼─────┐
    │  Scheduler  │ │ Webserver │ │  Triggerer │
    │ (调度核心)  │ │  (UI/API) │ │ (async感知)│
    └──────┬──────┘ └───────────┘ └───────────┘
           │
    ┌──────▼──────┐
    │   Executor  │
    │ (任务分发)  │
    └──────┬──────┘
           │
    ┌──────▼──────────────────────┐
    │         Workers             │
    │  (实际执行 task 的进程/Pod) │
    └─────────────────────────────┘
```

### 3.2 Executor 对比

| Executor | 并发模型 | 适用场景 | 优点 | 缺点 |
|----------|----------|----------|------|------|
| **SequentialExecutor** | 单进程串行 | 本地开发/测试 | 零配置 | 无并发 |
| **LocalExecutor** | 多进程 | 单机中小规模 | 简单、不需要额外组件 | 受限于单机资源 |
| **CeleryExecutor** | 分布式 Worker | 生产环境（传统） | 成熟稳定、水平扩展 | 需要 Redis/RabbitMQ broker |
| **KubernetesExecutor** | 每 task 一个 Pod | 云原生生产环境 | 资源隔离、弹性伸缩、按需分配 | 冷启动延迟、K8s 运维复杂度 |
| **CeleryKubernetesExecutor** | 混合 | 大规模异构 | 常规 task 用 Celery，特殊 task 用 K8s Pod | 最复杂的运维 |

> **Senior 级追问**：KubernetesExecutor 的 Pod 冷启动问题怎么缓解？
> 答：1) 预热 Pod Pool 2) 用轻量 base image 3) 对时间敏感 task 用 CeleryKubernetesExecutor 混合模式 4) 利用 Airflow 3.0 的 Edge Worker

### 3.3 HA Scheduler (Airflow 2.0+)

- 支持多个 Scheduler 实例 active-active
- 通过 metadata DB 行锁做协调
- 任一 Scheduler 挂掉，其他自动接管
- 生产环境建议至少 2 个 Scheduler

---

## 4. Airflow 2.x → 3.0 演进

### 4.1 Airflow 2.x 重要特性

| 版本 | 关键特性 |
|------|----------|
| 2.0 | TaskFlow API, HA Scheduler, 独立 Provider 包 |
| 2.1 | Cross-DAG dependencies via `ExternalTaskMarker` |
| 2.2 | Custom Timetable, Deferrable Operators |
| 2.3 | Dynamic Task Mapping (`expand()`) |
| 2.4 | Data-aware scheduling (Datasets, 后改名 Asset) |
| 2.5 | 进一步优化 Datasets, 改进 Listener |
| 2.6-2.10 | Asset 改进, 更多 provider 更新, UI 改进 |

### 4.2 Airflow 3.0 核心变化 ⭐

| 特性 | 说明 | 面试怎么说 |
|------|------|------------|
| **Asset-based Scheduling** | DAG 之间通过数据资产 (Asset) 触发，取代纯 cron 依赖 | "让编排从 time-driven 转向 event/data-driven" |
| **Backfill API** | 更好的回填控制（API 级别，非仅 CLI） | "生产中回填更安全可控" |
| **Edge Worker** | Worker 运行在不同网络/区域，Scheduler 远程分发 | "支持 hybrid cloud / edge computing 场景" |
| **改进 REST API** | 更完整的 API 覆盖 | "更容易与 CI/CD 和外部系统集成" |
| **Event-driven 方向** | 从轮询模型向事件驱动靠拢 | "减少延迟，提高资源利用率" |
| **Triggerer 增强** | 更多 deferrable operator 支持 | "长等待 task 不占 worker slot" |
| **TaskFlow 增强** | 更好的装饰器支持和类型提示 | "代码更简洁，IDE 支持更好" |

### 4.3 Asset-based Scheduling 详解

```python
from airflow.sdk import Asset, DAG, task

# 定义数据资产
my_dataset = Asset("s3://bucket/cleaned_data/")

# 生产者 DAG：产出 asset
@dag(schedule="@daily")
def producer():
    @task(outlets=[my_dataset])  # 声明产出
    def clean_data():
        # ... 清洗逻辑 ...
        pass

# 消费者 DAG：当 asset 更新时自动触发
@dag(schedule=[my_dataset])  # asset 作为 schedule
def consumer():
    @task
    def process_cleaned_data():
        # ... 处理逻辑 ...
        pass
```

> **面试亮点**："Asset-based scheduling 解耦了 DAG 之间的 time coupling。生产者不需要知道消费者是谁，消费者只声明依赖哪个 asset。这是 data mesh 和 data contract 思想在编排层的体现。"

---

## 5. 实战场景题 (Scenario Questions)

### 5.1 失败重试与告警

```python
default_args = {
    "retries": 3,
    "retry_delay": timedelta(minutes=5),
    "retry_exponential_backoff": True,   # 指数退避
    "max_retry_delay": timedelta(minutes=30),
    "on_failure_callback": alert_on_failure,  # 自定义告警
    "on_retry_callback": log_retry,
    "sla": timedelta(hours=2),  # SLA 超时告警
    "execution_timeout": timedelta(hours=1),  # 单 task 超时
}

def alert_on_failure(context):
    ti = context["task_instance"]
    dag_id = context["dag"].dag_id
    # 发 Slack / PagerDuty / Email
    send_alert(f"Task {ti.task_id} in {dag_id} failed!")
```

| 参数 | 作用 | 推荐值 |
|------|------|--------|
| `retries` | 最大重试次数 | 2-3（避免无限重试） |
| `retry_delay` | 重试间隔 | 5-10 min |
| `retry_exponential_backoff` | 指数退避 | `True`（避免打爆外部系统） |
| `execution_timeout` | 单 task 运行上限 | 根据实际预期 × 2-3 |
| `dagrun_timeout` | 整个 DAG Run 上限 | 在 DAG 级别设置 |
| `sla` | 未在预期时间完成则告警 | 设置合理的业务 SLA |

### 5.2 幂等性 (Idempotency) ⭐⭐⭐

> **这是 Senior 级面试必考题。**

**原则**：同一个 task 对同一个 `logical_date` 重跑 N 次，结果应该完全一样。

| 场景 | 非幂等写法 ❌ | 幂等写法 ✅ |
|------|-------------|------------|
| 写入表 | `INSERT INTO target SELECT ...` | `DELETE WHERE date=... THEN INSERT` 或 `MERGE/UPSERT` |
| 分区表 | `APPEND` 模式 | `OVERWRITE PARTITION(date=...)` |
| 文件输出 | 随机文件名 | 用 `logical_date` 构造确定性路径 |
| API 调用 | 无去重 | 用幂等 key / 先检查再写入 |
| 发通知 | 每次重跑都发 | 检查是否已发过 |

```python
@task
def load_data(logical_date=None):
    partition = logical_date.strftime("%Y-%m-%d")
    # ✅ 幂等：覆盖写入特定分区
    spark.sql(f"""
        INSERT OVERWRITE TABLE target PARTITION (dt='{partition}')
        SELECT * FROM staging WHERE dt = '{partition}'
    """)
```

### 5.3 Dynamic Task Mapping (Airflow 2.3+)

```python
@task
def get_file_list() -> list[str]:
    return ["file1.csv", "file2.csv", "file3.csv"]

@task
def process_file(filename: str):
    print(f"Processing {filename}")

# expand() 动态创建 N 个 task instance
files = get_file_list()
process_file.expand(filename=files)
```

| 适用场景 | 说明 |
|----------|------|
| 文件列表处理 | 每个文件一个 task，并行处理 |
| 分区回填 | 每个分区一个 task |
| 多表 ETL | 每张表一个 task |
| API 分页 | 每页一个 task |

> **面试答法**："Dynamic Task Mapping 替代了以前在 DAG 解析时用 for 循环生成 task 的方式。区别在于 for 循环是解析时确定数量，而 expand 是运行时动态确定，更灵活也更安全。"

### 5.4 数据质量检查

```python
@task
def check_data_quality(logical_date=None):
    partition = logical_date.strftime("%Y-%m-%d")
    
    # 行数检查
    count = db.execute(f"SELECT COUNT(*) FROM target WHERE dt='{partition}'")
    assert count > 0, f"No data for {partition}!"
    
    # NULL 比例检查
    null_ratio = db.execute(f"""
        SELECT COUNT(*) FILTER (WHERE key_col IS NULL) * 1.0 / COUNT(*)
        FROM target WHERE dt='{partition}'
    """)
    assert null_ratio < 0.05, f"NULL ratio {null_ratio} exceeds 5% threshold!"
    
    # 重复检查
    dup_count = db.execute(f"""
        SELECT COUNT(*) - COUNT(DISTINCT id) 
        FROM target WHERE dt='{partition}'
    """)
    assert dup_count == 0, f"Found {dup_count} duplicates!"

# DAG 中的位置
extract() >> transform() >> load() >> check_data_quality()
```

> **进阶**：生产中推荐用 Great Expectations 或 dbt tests 替代手写 assert，配合 `BranchPythonOperator` 做条件处理。

### 5.5 Backfill 策略

```bash
# CLI 回填
airflow dags backfill \
    --start-date 2026-01-01 \
    --end-date 2026-03-01 \
    --reset-dagruns \       # 清除已有 DAG Run
    --rerun-failed-tasks \  # 只重跑失败 task
    my_dag
```

| 策略 | 做法 | 原因 |
|------|------|------|
| 限制并发 | 设置 `max_active_runs=3` | 避免同时跑 90 个 DAG Run 压垮系统 |
| 用 Pool | 关键外部资源用 Pool 限制 | 保护数据库、API |
| 分批回填 | 一次回填一周而非一年 | 更可控，出错易排查 |
| 确保幂等 | OVERWRITE 而非 APPEND | 回填本质是重跑 |
| 监控 | 关注 scheduler 负载、DB 连接数 | 回填是资源密集操作 |

---

## 6. 生产最佳实践 (Best Practices)

### 6.1 DAG 编写规范

| 规则 | 说明 |
|------|------|
| DAG 文件轻量 | 不要在顶层做 import heavy 库、DB 查询、API 调用 |
| 一个文件一个 DAG | 便于管理和 CI/CD |
| 用 `default_args` | 统一 retry, callback, owner 等配置 |
| 显式设置 `start_date` | 不要用 `datetime.now()`，会导致每次 parse 都变 |
| `catchup=False` | 除非你真的需要回填历史 |
| `tags` | 方便 UI 过滤，如 `["team-data", "priority-high"]` |
| `doc_md` | 给 DAG 写文档，在 UI 里显示 |
| 用 Jinja 模板 | `{{ ds }}`, `{{ logical_date }}` 等宏，保持幂等 |

### 6.2 测试策略

```python
# 1. DAG 加载测试（CI 必备）
def test_dag_loads():
    from my_dag_file import my_dag
    assert my_dag is not None
    assert len(my_dag.tasks) > 0

# 2. DAG 完整性测试
def test_no_import_errors():
    from airflow.models import DagBag
    dag_bag = DagBag(include_examples=False)
    assert len(dag_bag.import_errors) == 0

# 3. Task 逻辑单元测试
def test_transform_logic():
    result = transform_function({"data": [1, 2, 3]})
    assert result == {"data": [2, 4, 6]}
```

### 6.3 部署模式

| 模式 | 工具 | 适用 |
|------|------|------|
| **Managed Service** | Astronomer / MWAA (AWS) / Cloud Composer (GCP) | 不想运维 Airflow 本身 |
| **Helm Chart** | `apache/airflow` Helm chart on K8s | 自运维但要灵活 |
| **Docker Compose** | 官方 `docker-compose.yaml` | 本地开发 / 小团队 |

---

## 7. 反模式 (Anti-Patterns) ⚠️

| 反模式 | 问题 | 正确做法 |
|--------|------|----------|
| XCom 传大数据 | metadata DB 爆炸、序列化慢 | 写到 S3/GCS，XCom 只传路径 |
| DAG 文件顶层做重计算 | scheduler parse 卡顿 | 把逻辑放在 task 内部 |
| 隐式依赖 | task A 写表、task B 读表但无显式依赖 | 用 `>>` 显式声明 |
| Mega-DAG | 一个 DAG 几百个 task | 拆分为多个 DAG + Asset/TriggerDagRun |
| `start_date=datetime.now()` | 每次 parse 都变，导致调度混乱 | 用固定日期 |
| 在 DAG 中硬编码密码 | 安全风险 | 用 Connection / Secrets Backend |
| 不设 timeout | task 可能无限运行 | 设 `execution_timeout` |
| Sensor 用 poke 模式长等待 | 占着 worker slot 浪费资源 | 长等待用 `mode="reschedule"` 或 Deferrable |
| 不设 Pool | 所有 task 同时打外部系统 | 用 Pool 限制并发 |

---

## 8. Airflow vs 竞品对比

| 维度 | Airflow | Prefect | Dagster | dbt |
|------|---------|---------|---------|-----|
| **定位** | 通用工作流编排 | 现代工作流编排 | Software-defined assets | SQL transformation 层 |
| **核心抽象** | DAG + Operator | Flow + Task (无 DAG 概念) | Asset + Op + Job | Model + Source + Test |
| **调度** | Cron / Timetable / Asset | Cron / Event | Cron / Sensor / Asset | 无（靠外部编排） |
| **执行** | Push-based (Scheduler 推) | Hybrid (Agent pull) | Push-based | CLI / Cloud API |
| **数据感知** | 3.0 Asset-based | Limited | 原生 asset lineage | 原生 lineage |
| **学习曲线** | 中等 | 低（更 Pythonic） | 中高（概念多） | 低（纯 SQL） |
| **社区生态** | 最大，Provider 最多 | 成长中 | 成长中 | 非常活跃 |
| **典型搭配** | Airflow 编排 + dbt 做 transform | 替代 Airflow | 替代 Airflow | 被 Airflow/Dagster 编排 |

### Airflow 编排 dbt 的常见模式

| 模式 | 做法 | 优缺点 |
|------|------|--------|
| `BashOperator` | `dbt run --select model_name` | 最简单，但粒度粗 |
| **Cosmos Provider** | `DbtTaskGroup` 自动把每个 model 变成一个 task | 细粒度、可视化好、推荐 |
| dbt Cloud API | `DbtCloudRunJobOperator` | 如果用 dbt Cloud |

---

## 9. 常见面试问答 (Q&A)

### Q1: Airflow 适合做什么？不适合做什么？

**适合**：批处理编排、ETL/ELT 调度、ML pipeline 编排、跨系统依赖管理
**不适合**：实时流处理（用 Kafka/Flink）、亚秒级调度、数据处理本身（Airflow 是编排层不是计算引擎）

### Q2: 如何处理跨 DAG 依赖？

1. `ExternalTaskSensor` — 等待另一个 DAG 的 task 完成
2. `TriggerDagRunOperator` — 主动触发下游 DAG
3. **Asset-based scheduling (3.0)** — 通过数据资产自动触发（推荐）

### Q3: Deferrable Operator 是什么？解决什么问题？

传统 Sensor 在 poke 模式下占着 worker slot 等待。Deferrable Operator 把等待逻辑交给 **Triggerer**（独立的轻量进程），释放 worker slot。等条件满足后 Triggerer 通知 Scheduler 恢复执行。大幅减少 worker 资源浪费。

### Q4: 你会怎么设计一个日增量 ETL pipeline？

```
[S3Sensor] → [Extract: 读增量文件]
                    ↓
            [Transform: 清洗 + 标准化]
                    ↓
            [Load: MERGE/UPSERT 到目标表]
                    ↓
            [Quality Check: 行数/NULL/重复]
                    ↓
            [Notify: 成功/失败通知]
```

关键设计点：
- 用 `logical_date` 构造文件路径 → 幂等
- Load 用 MERGE 而非 INSERT → 幂等
- Quality Check 失败 → 标记 task failed → 触发告警
- 设 Pool 保护目标数据库
- `max_active_runs=1` 防止并发写入冲突

### Q5: Airflow metadata DB 性能问题怎么处理？

- 定期清理历史 DAG Run 和 Task Instance（`airflow db clean`）
- XCom 不存大数据
- 适当增大 `min_file_process_interval` 减少 parse 频率
- 用 PostgreSQL 而非 SQLite/MySQL
- 合理设置 connection pool size

### Q6: 你用过 Airflow 解决什么实际问题？

> **提示**：结合你自己的项目经验，比如 NAAAP 的 dbt + BigQuery pipeline、日记自动化 Lambda pipeline 等，讲你用（或会用）Airflow 编排的场景。面试官看的是你能不能把 Airflow 概念映射到真实问题。

---

## 10. 速记卡片 (Quick Reference)

### 核心 CLI 命令

```bash
airflow dags list                          # 列出所有 DAG
airflow dags trigger my_dag                # 手动触发
airflow dags backfill -s ... -e ... my_dag # 回填
airflow tasks test my_dag my_task 2026-03-30  # 测试单个 task（不记录 DB）
airflow tasks run my_dag my_task 2026-03-30   # 运行单个 task（记录 DB）
airflow db check                           # 检查 DB 连接
airflow db clean --clean-before-timestamp ...  # 清理历史数据
airflow connections list                   # 列出连接
airflow variables get my_var               # 获取变量
```

### 核心配置参数

| 参数 | 说明 | 推荐值 |
|------|------|--------|
| `parallelism` | 全局最大并发 task 数 | 根据 worker 资源设 |
| `max_active_tasks_per_dag` | 单 DAG 最大并发 task | 16-32 |
| `max_active_runs_per_dag` | 单 DAG 最大并发 DAG Run | 1-3（看场景） |
| `min_file_process_interval` | DAG 文件扫描间隔（秒） | 30-60 |
| `dag_dir_list_interval` | DAG 目录扫描间隔（秒） | 300 |
| `worker_concurrency` (Celery) | 每个 Worker 并发数 | 8-16 |

### Jinja 模板常用宏

| 宏 | 值 | 示例 |
|----|----|------|
| `{{ ds }}` | `YYYY-MM-DD` (logical_date) | `2026-03-30` |
| `{{ ds_nodash }}` | `YYYYMMDD` | `20260330` |
| `{{ logical_date }}` | 完整 datetime 对象 | |
| `{{ data_interval_start }}` | 数据区间起点 | |
| `{{ data_interval_end }}` | 数据区间终点 | |
| `{{ ts }}` | ISO format timestamp | |
| `{{ macros.ds_add(ds, 7) }}` | 日期加减 | `2026-04-06` |
| `{{ params.my_param }}` | 自定义参数 | |

---

> 💡 **面试最终 Tips**：
> 1. 不要只背概念 —— 每个知识点都准备一个"我在项目中怎么用的"故事
> 2. 提到 Airflow 3.0 Asset-based scheduling 是加分项
> 3. 幂等性 + 失败处理是 Senior 级必考，准备具体例子
> 4. 能说出反模式 = 有实战经验的信号
> 5. 知道 Airflow 的边界（不适合做什么）比知道它能做什么更重要